# Student Performance Prediction

Post-graduate project — EDA, Regression & Classification

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline

## 1. Load & Explore

In [ ]:
from src.data_preprocessing import load_data
df = load_data()
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df.isnull().sum()

## 2. Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
cols = ['reading_score', 'writing_score', 'math_score', 'gender', 'lunch', 'test_prep']
for ax, col in zip(axes.flat, cols):
    if df[col].dtype == 'object':
        df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=45)
    else:
        df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='black')
        ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['grade'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Grade Distribution')
df[['reading_score', 'writing_score', 'math_score']].boxplot(ax=axes[1])
axes[1].set_title('Score Distributions')
plt.tight_layout()
plt.show()

## 3. Correlations

In [ ]:
num_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
sns.pairplot(df, vars=['reading_score', 'writing_score', 'math_score'], diag_kind='kde', hue='grade')
plt.show()

## 4. Categorical Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cats = ['gender', 'race_ethnicity', 'lunch', 'test_prep']
for ax, col in zip(axes.flat, cats):
    df.groupby(col)['math_score'].mean().sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Average Math Score by {col}')
    ax.set_xlabel('Mean Math Score')
plt.tight_layout()
plt.show()

## 5. Model Training

In [ ]:
from src.data_preprocessing import preprocess, split_data
from src.model_training import train_models

reg_results, clf_results = train_models()

In [ ]:
reg_summary = pd.DataFrame([
    {'model': k, 'MAE': v['mae'], 'R2': v['r2']} for k, v in reg_results.items()
])
reg_summary

In [ ]:
clf_summary = pd.DataFrame([
    {'model': k, 'Accuracy': v['accuracy']} for k, v in clf_results.items()
])
clf_summary

## 6. Evaluation Plots

In [ ]:
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from src.evaluation import plot_residuals, plot_confusion, plot_feature_importance
from src.data_preprocessing import load_data, preprocess, split_data

df = load_data()
X, y_reg, y_clf, encoders, scaler = preprocess(df)
result = split_data(X, y_reg, y_clf)
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = result

reg = joblib.load(Path('..') / 'models' / 'regression_model.pkl')
clf = joblib.load(Path('..') / 'models' / 'classification_model.pkl')

y_reg_pred = reg.predict(X_test)
y_clf_pred = clf.predict(X_test)

plot_residuals(y_reg_test, y_reg_pred, 'random_forest_regressor')
plot_confusion(y_clf_test, y_clf_pred, 'random_forest_classifier')
plot_feature_importance(reg, X.columns, 'random_forest_regressor')

## 7. Make a Prediction

In [ ]:
from src.predict import StudentPerformancePredictor

predictor = StudentPerformancePredictor()

sample = {
    'gender': 'female',
    'race_ethnicity': 'group B',
    'parental_education': "bachelor's",
    'lunch': 'standard',
    'test_prep': 'completed',
    'reading_score': 85,
    'writing_score': 82,
}
result = predictor.predict(sample)
print(result)